We define the states, transition matrix, and emission matrix as specified in the Nature Primer.

In [1]:
import math

# Define transition probabilities (log domain)
trans = {
    'E':  {'E': math.log(0.9), '5': math.log(0.1), 'I': -math.inf, 'End': -math.inf},
    '5':  {'E': -math.inf,     '5': -math.inf,     'I': math.log(1.0), 'End': -math.inf},
    'I':  {'E': -math.inf,     '5': -math.inf,     'I': math.log(0.9), 'End': math.log(0.1)}
}

# Define emission probabilities (log domain)
emit = {
    'E': {'A': math.log(0.25), 'C': math.log(0.25), 'G': math.log(0.25), 'T': math.log(0.25)},
    '5': {'A': math.log(0.05), 'C': -math.inf,      'G': math.log(0.95), 'T': -math.inf},
    'I': {'A': math.log(0.4),  'C': math.log(0.1),  'G': math.log(0.1),  'T': math.log(0.4)}
}



We implement get_log_prob_of_a_given_path() to compute the log-probability of a specific state path for a DNA sequence.

In [2]:
def get_log_prob_of_a_given_path(path, sequence):

    # Initialize log probability
    log_prob = 0.0

    # Initial state (Start -> First state in path)
    first_state = path[0]
    log_prob += math.log(1.0)  # Start -> E (forced by model)
    log_prob += emit[first_state][sequence[0]]  # First emission

    # Iterate through the rest of the path
    for i in range(1, len(path)):
        prev_state = path[i-1]
        curr_state = path[i]
        log_prob += trans[prev_state][curr_state]  # Transition
        log_prob += emit[curr_state][sequence[i]]  # Emission

    # Transition to End (if last state is 'I')
    if path[-1] == 'I':
        log_prob += math.log(0.1)  # I -> End

    return log_prob


In [3]:
# Example from the primer
path = "EEEEEEEEEEEEEEEEEE5IIIIIII"
sequence ="CTTCATGTGAAAGCAGACGTAAGTCA"
print(get_log_prob_of_a_given_path(path, sequence))

-41.21967768602254


We implement the Viterbi algorithm to find the most likely state path that emits the observed DNA sequence.

In [4]:
def viterbi(sequence):
    # Define states and emissions
    states = ['E', '5', 'I']
    emissions = ['A', 'C', 'G', 'T']

    # Transition probabilities (log space)
    trans = {
        'Start': {'E': math.log(1.0), '5': -math.inf, 'I': -math.inf, 'End': -math.inf},
        'E': {'E': math.log(0.9), '5': math.log(0.1), 'I': -math.inf, 'End': -math.inf},
        '5': {'E': -math.inf, '5': -math.inf, 'I': math.log(1.0), 'End': -math.inf},
        'I': {'E': -math.inf, '5': -math.inf, 'I': math.log(0.9), 'End': math.log(0.1)}
    }

    # Emission probabilities (log space)
    emit = {
        'E': {'A': math.log(0.25), 'C': math.log(0.25), 'G': math.log(0.25), 'T': math.log(0.25)},
        '5': {'A': math.log(0.05), 'C': -math.inf, 'G': math.log(0.95), 'T': -math.inf},
        'I': {'A': math.log(0.4), 'C': math.log(0.1), 'G': math.log(0.1), 'T': math.log(0.4)}
    }

    # Initialize Viterbi matrix and backpointer
    V = [{}]
    backpointer = [{}]

    # Initialization step (Start state -> 'E')
    for s in states:
        V[0][s] = trans['Start'][s] + emit[s][sequence[0]] if s == 'E' else -math.inf
        backpointer[0][s] = 'Start'

    # Recursion step (for each subsequent state)
    for t in range(1, len(sequence)):
        V.append({})
        backpointer.append({})
        for s in states:
            max_prob = -math.inf
            best_prev = None
            for prev_s in states:
                prob = V[t-1][prev_s] + trans[prev_s][s] + emit[s][sequence[t]]
                if prob > max_prob:
                    max_prob = prob
                    best_prev = prev_s
            V[t][s] = max_prob
            backpointer[t][s] = best_prev

    # Termination step (final transition to 'End')
    max_final = -math.inf
    best_final_state = None
    for s in states:
        prob = V[-1][s] + trans[s]['End']
        if prob > max_final:
            max_final = prob
            best_final_state = s

    # Backtracking (from 'End' to Start)
    best_path = []
    current_state = best_final_state
    best_path.append(current_state)
    for t in range(len(sequence) - 1, 0, -1):
        best_path.insert(0, backpointer[t][current_state])
        current_state = backpointer[t][current_state]

    return max_final, ''.join(best_path)

In [5]:
sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"
viterbi_prob, viterbi_path = viterbi(sequence)
print("Most likely path:", viterbi_path)

Most likely path: EEEEEEEEEEEEEEEEEE5IIIIIII


Note:We define transition and emission probabilities in log-space to avoid numerical underflow during computations.